In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
try:
    plt.style.use(['science', 'ieee', 'no-latex']) # 使用 no-latex 以确保无需本地 tex 环境也能运行
except:
    print("未检测到 SciencePlots，将使用默认样式。建议 pip install SciencePlots")
    plt.style.use('seaborn-v0_8-white')

未检测到 SciencePlots，将使用默认样式。建议 pip install SciencePlots


In [2]:
def generate_refined_figures():
    # -------------------------------------------------------------------------
    # 通用设置
    # -------------------------------------------------------------------------
    # 柔化后的红色 (Soft Coral / Pastel Red)
    soft_red_hex = '#E57373'  
    # 2D背景的超浅红色 (用于表示广阔的低似然空间)
    bg_red_hex = '#FFEBEE'
    
    # -------------------------------------------------------------------------
    # 1. 生成 3D 图 (柔和版)
    # -------------------------------------------------------------------------
    # 3D图保持中心在原点，因为它是一个独立的素材，不需要偏置
    mu_3d = [0, 0]
    cov_3d = [[3.0, 1.5], [1.5, 3.0]] 
    
    x = np.linspace(-6, 6, 300)
    y = np.linspace(-6, 6, 300)
    X, Y = np.meshgrid(x, y)
    pos = np.dstack((X, Y))
    
    rv_3d = multivariate_normal(mu_3d, cov_3d)
    Z_3d = rv_3d.pdf(pos)
    rho_3d = Z_3d.max() * 0.08
    
    fig1 = plt.figure(figsize=(6, 6))
    ax1 = fig1.add_subplot(111, projection='3d')
    
    # --- 颜色映射逻辑优化 ---
    color_array = np.zeros(Z_3d.shape + (4,))
    mask_low = Z_3d < rho_3d
    
    # 1. 低似然区域 (柔和红)
    # 使用更柔和的红色，并稍微降低不透明度
    color_array[mask_low] = mcolors.to_rgba(soft_red_hex, alpha=0.6) 
    
    # 2. 高似然区域 (渐变蓝)
    z_norm = (Z_3d - rho_3d) / (Z_3d.max() - rho_3d)
    z_norm = np.clip(z_norm, 0, 1)
    blues = plt.get_cmap('Blues')
    # 核心区域保持蓝色，但在边缘处平滑过渡
    colors_high = blues(z_norm * 0.7 + 0.3) 
    color_array[~mask_low] = colors_high[~mask_low]
    color_array[~mask_low, 3] = 0.8 # 核心区域稍微实一点

    surf = ax1.plot_surface(X, Y, Z_3d, 
                            facecolors=color_array,
                            linewidth=0, 
                            antialiased=True, 
                            shade=False) # 关闭光照，保持纯色更干净

    ax1.set_axis_off()
    ax1.view_init(elev=35, azim=-60)
    
    plt.savefig('gaussian_3d_soft.png', transparent=True, dpi=300, bbox_inches='tight', pad_inches=0)
    plt.close()


    # -------------------------------------------------------------------------
    # 2. 生成 2D 图 (偏移版 - 为左上角留空)
    # -------------------------------------------------------------------------
    fig2, ax2 = plt.figure(figsize=(6, 6)), plt.gca()
    
    # --- 关键修改：偏移均值 ---
    # 将分布中心移到右下角 (x正, y负)
    # 这样左上角 (x负, y正) 就会空出来
    mu_2d = [2., -2.] 
    cov_2d = [[3.0, 1.5], [1.5, 3.0]] # 保持形状不变
    
    rv_2d = multivariate_normal(mu_2d, cov_2d)
    Z_2d = rv_2d.pdf(pos)
    rho_2d = Z_2d.max() * 0.08

    # --- 绘制等高线 ---
    
    # 1. 边缘区域背景 (Sampling Space)
    # 填充整个低似然背景，但颜色非常浅，不干扰前景
    ax2.contourf(X, Y, Z_2d, levels=[0, rho_2d], colors=[bg_red_hex], alpha=0.5)
    
    # 2. 核心区域 (Bias)
    # 蓝色渐变
    levels_high = np.linspace(rho_2d, Z_2d.max(), 10)
    ax2.contourf(X, Y, Z_2d, levels=levels_high, cmap='Blues', alpha=0.8)
    
    # 3. 边界线 (Cutoff)
    # 使用柔和红的实线或虚线
    ax2.contour(X, Y, Z_2d, levels=[rho_2d], colors=[soft_red_hex], linewidths=2.0, linestyles='--')
    
    # --- 移除坐标轴 ---
    ax2.set_axis_off()
    ax2.set_aspect('equal')
    
    # 确保保存范围覆盖 [-6, 6]，这样分布就在右下角，而不是自动裁剪居中
    ax2.set_xlim(-6, 6)
    ax2.set_ylim(-6, 6)
    
    plt.savefig('gaussian_2d_shifted.png', transparent=True, dpi=300, bbox_inches='tight', pad_inches=0)
    plt.close()

    print("图片生成完毕：")
    print("1. gaussian_3d_soft.png (红色更柔和)")
    print("2. gaussian_2d_shifted.png (分布位于右下角，左上角留白)")

In [3]:
if __name__ == "__main__":
    generate_refined_figures()

图片生成完毕：
1. gaussian_3d_soft.png (红色更柔和)
2. gaussian_2d_shifted.png (分布位于右下角，左上角留白)
